In [0]:
import requests
import json
from datetime import datetime
from infos_credenciais import KEY
#INTENSIDADE DE CARBONO LIBERADO EM KWH POR MES E DEMANDA DE ELETRICIDADE POR MES EM TWH

def extrair_carbono_eletricidade(path, indicador, nome_volume, data_inicio=None):

  dados_extraidos = []
  api_key = f"{KEY}"
  base_url = "https://api.ember-energy.org"
  query_url = (
    f"{base_url}/v1/{path}" + f"?&api_key={api_key}")
  
  if data_inicio:
    query_url += f"&start_date={data_inicio}"
  else:
    print("buscando carga histórica completa")

  response = requests.get(query_url)
  response.raise_for_status()
  json_all = response.json()
  json_data = json_all.get('data', [])

  for x in json_data:
    dados_extraidos.append({
    'country': x.get('entity'),
    'iso3': x.get('entity_code'),
    'date': x.get('date'),
    'metric_name': indicador,  
    'value': float(x.get(indicador)) if x.get(indicador) is not None else None        
    })

  if not dados_extraidos:
        print(f"nenhum dado novo encontrado para o indicador {indicador}. Arquivo não gerado.")
        return # sai da função sem executar o with open abaixo
      

  timestamp_extracao = datetime.now().strftime("%Y%m%d_%H%M%S")

  pasta_destino = f"/Volumes/portfolio_energia/bronze/landing/{nome_volume}"
  os.makedirs(pasta_destino, exist_ok=True) # serve para criar um "subdiretório" dentro de um mesmo volume

  caminho_arquivo = f"{pasta_destino}/extracao_{timestamp_extracao}.json"
  with open(caminho_arquivo, "w", encoding="utf-8") as arquivo_json:
    json.dump(dados_extraidos, arquivo_json, ensure_ascii=False)
    #aqui, o with open abre um arquivo em branco a partir do caminho arquivo, e a instrução W(write),

def obter_ultima_data(nome_tabela_bronze):
  if spark.catalog.tableExists(nome_tabela_bronze):
    return spark.sql(f"SELECT MAX(date) FROM {nome_tabela_bronze}").collect()[0][0]
  else:
    return None

data_carbono = obter_ultima_data("portfolio_energia.bronze.ember_intensity")
extrair_carbono_eletricidade("carbon-intensity/monthly", "emissions_intensity_gco2_per_kwh", "landing_ember_carbon", data_inicio=data_carbono)

data_eletricidade = obter_ultima_data("portfolio_energia.bronze.ember_demand")
extrair_carbono_eletricidade("electricity-demand/monthly", "demand_twh", "landing_ember_electricity", data_inicio=data_eletricidade)

In [0]:
from pyspark.sql.functions import current_timestamp

def rodar_autoloader_ember(nome_volume, tabela_destino):
    caminho_base = f"/Volumes/portfolio_energia/bronze/landing/{nome_volume}"
    df_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{caminho_base}/_schema")
        .load(f"{caminho_base}/extracao_*.json")
    )
    
    df_stream_com_data = df_stream.withColumn("data_ingestao_bronze", current_timestamp())
                                              

    query = (
        df_stream_com_data.writeStream
        .format("delta")
        .option("checkpointLocation", f"{caminho_base}/_checkpoint")
        .trigger(availableNow=True) # processa tudo o que chegou de novo e desliga
        .toTable(f"portfolio_energia.bronze.{tabela_destino}")
    )
    
    query.awaitTermination() # espera o stream terminar de gravar antes de ir para a próxima linha do código

rodar_autoloader_ember("landing_ember_carbon", "ember_intensity")
rodar_autoloader_ember("landing_ember_electricity", "ember_demand")